In [2]:
import pandas as pd

In [5]:
df = pd.read_excel('AQR_2026_ER_SER.xlsx',sheet_name='ER vs SER data')
label_chains = list(df['Label Chain Name'].unique())
label_chains.sort()
data = {'label chain':[],'# evals':[],'% audited':[],'% Egregious audited evals':[],'% Poor audited evals':[],'% Okay audited evals':[],'% Good audited evals':[]}
for lc in label_chains:
    data['label chain'].append(lc)
    tmp = df[df['Label Chain Name']==lc]
    evals = list(tmp['EncryptedString'].unique())
    data['# evals'].append(len(evals))
    audited_evals = list(tmp[tmp['Is Audited By Ser']]['EncryptedString'].unique())
    data['% audited'].append(len(audited_evals)/len(evals))

    for a in ['Egregious','Poor','Okay','Good']:
        n_a_evals = len(list(tmp[tmp['Ser Consensus Answer']==a]['EncryptedString'].unique()))
        perc = n_a_evals/len(audited_evals) if len(audited_evals) else 0
        data['% '+a+' audited evals'].append(perc)

data_df = pd.DataFrame(data)
data_df.to_excel('AQR_2026_ER_SER_label_chain_distributions.xlsx',index=False)




In [7]:
# Tailor AQR content for ER autorater project

# Load data
content = pd.read_excel('AQR eval content.xlsx')
er_ser  = pd.read_excel('AQR_2026_ER_SER.xlsx')

# Filter data
labels = ['Egregious','Poor','Okay','Good']
er_ser = er_ser[er_ser['Ser Consensus Answer'].isin(labels)]

# Create support columns
def is_correct(row):
    return 1.0 if row['Rater Answer'] == row['Ser Consensus Answer'] else 0.0
def ad_type(row):
    lc = str(row['Label Chain Name']).lower()
    return 'shopping' if 'shopping' in lc else ('text' if 'text' in lc else 'Not Found')
er_ser['is correct?'] = er_ser.apply(is_correct, axis=1)
er_ser['Ad Type'] = er_ser.apply(ad_type, axis=1)

# Build data set of evals' content + ad type + ER agreement % with SER
# All evals in said data set must've been rated by 3 Welo ERs
evals = list(content['EncryptedString'].unique())
data = {
    'EncryptedString':[],
    'Ad Type':[],
    'ER agreement % with SER':[],
    'ER Maj & SER Agree':[]
}
for e in evals:
    tmp = er_ser[er_ser['EncryptedString']==e]
    if len(tmp) >= 2:
        data['EncryptedString'].append(e)
        data['Ad Type'].append(str(tmp['Ad Type'].iloc[0]))
        ar = float(tmp['is correct?'].mean())
        data['ER agreement % with SER'].append(ar)
        data['ER Maj & SER Agree'].append('TRUE' if ar > 0.6 else 'FALSE')
    else:
        continue
data_df = pd.DataFrame(data)
w3_data = pd.merge(content, data_df, on='EncryptedString', how='left')

# Save data
w3_data.to_excel('AQR eval content-type-agreement (2 or 3 Welo ER).xlsx',index=False)

# report numbers
print('# evals with content:', len(content))
print('# evals with content & rated by 3 Welo ERs:', len(data_df))

# evals with content: 12066
# evals with content & rated by 3 Welo ERs: 4593
